In [27]:
import torch.nn as nn
import numpy as np
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

In [28]:
VOCAB_SIZE  = 30000
SEQ_LEN     = 8      # T — context window; input sequence is (T, D_MODEL) = (8, 8)
D_MODEL     = 8      # input hidden dim

# ── MLA-specific dims ─────────────────────────────────────────────────────────
N_HEADS      = 2     # number of Q/K heads
D_HEAD_NOPE  = 4     # NoPE (content) dim per head
D_HEAD_ROPE  = 2     # RoPE (positional) dim per head — small, stays uncompressed
D_HEAD       = D_HEAD_NOPE + D_HEAD_ROPE   # 6: effective full head dim for attention
D_C_KV       = 4    # KV latent compression dim  →  W_DKV: (D_MODEL × D_C_KV) = (4×4)
D_C_Q        = 4    # Q  latent compression dim  →  W_DQ:  (D_MODEL × D_C_Q)  = (4×4)

D_FF        = 16
DROPOUT     = 0.1
BATCH_SIZE  = 64
LR          = 3e-4
EPOCHS      = 1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")
print(f"input shape per sequence: ({SEQ_LEN}, {D_MODEL})")
print(f"W_DKV shape: ({D_MODEL}, {D_C_KV})")

device: cuda
input shape per sequence: (8, 8)
W_DKV shape: (8, 4)


In [29]:
TRAIN_BIN = Path.cwd() / "data" / "train.bin"

token_ids = np.fromfile(TRAIN_BIN, dtype=np.uint16).astype(np.int64)
print(f"total tokens: {len(token_ids):,}")

class TokenDataset(Dataset):
    def __init__(self, ids, seq_len):
        self.ids = torch.tensor(ids, dtype=torch.long)
        self.seq_len = seq_len

    def __len__(self):
        return len(self.ids) - self.seq_len

    def __getitem__(self, idx):
        x = self.ids[idx : idx + self.seq_len]
        y = self.ids[idx + 1 : idx + self.seq_len + 1]
        return x, y

dataset    = TokenDataset(token_ids, SEQ_LEN)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
print(f"batches per epoch: {len(dataloader)}")

total tokens: 136,633
batches per epoch: 2134


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Decoupled RoPE utilities
# Applied ONLY to the small Q_rope / K_rope pathways, never to the NoPE content.
# ══════════════════════════════════════════════════════════════════════════════

def precompute_freqs_cis(d_rope: int, max_seq_len: int, theta: float = 10000.0):
    """
    Returns complex unit vectors for RoPE rotation.
    d_rope must be even.
    Output shape: (max_seq_len, d_rope // 2)  — complex64
    """
    assert d_rope % 2 == 0
    # inverse frequencies: θ_i = 1 / 10000^(2i / d_rope)
    inv_freq = 1.0 / (theta ** (torch.arange(0, d_rope, 2).float() / d_rope))
    t        = torch.arange(max_seq_len, dtype=torch.float32)
    # outer product → angles matrix  (max_seq_len, d_rope//2)
    angles   = torch.outer(t, inv_freq)
    # e^{i·angle} = cos + i·sin  — the rotation in complex form
    freqs_cis = torch.polar(torch.ones_like(angles), angles)
    return freqs_cis   # (T, d_rope//2)  complex64


def apply_rope(x: torch.Tensor, freqs_cis: torch.Tensor) -> torch.Tensor:
    """
    Rotate x using pre-computed complex frequencies.
    x         : (B, n_heads, T, d_rope)   — real
    freqs_cis : (T, d_rope // 2)          — complex
    Returns   : (B, n_heads, T, d_rope)   — real, rotated
    """
    B, H, T, d = x.shape
    # pair up consecutive dims as Re/Im of a complex number
    x_c = torch.view_as_complex(x.float().reshape(B, H, T, d // 2, 2))  # (B,H,T,d/2) complex
    # broadcast freqs over batch and heads
    f   = freqs_cis[:T].unsqueeze(0).unsqueeze(0)    # (1, 1, T, d/2) complex
    # multiply = rotate in 2D plane
    out = torch.view_as_real(x_c * f).reshape(B, H, T, d)
    return out.to(x.dtype)

# Pre-compute once for the full context length
freqs_cis = precompute_freqs_cis(D_HEAD_ROPE, SEQ_LEN).to(DEVICE)
print(f"freqs_cis shape: {freqs_cis.shape}  (T={SEQ_LEN}, d_rope/2={D_HEAD_ROPE//2})  dtype={freqs_cis.dtype}")

freqs_cis shape: torch.Size([8, 1])  (T=8, d_rope/2=1)  dtype=torch.complex64


# RoPE Demo

In [36]:
cis = precompute_freqs_cis(4, SEQ_LEN)
x = torch.ones(1, 1, SEQ_LEN, 4)  # (B,H,T,d_rope)
out = apply_rope(x, cis)
print("Input tensor:")
print(x)
print("\nPre-computed freqs_cis:")
print(cis)
print("\nRotated tensor:")
print(out)

Input tensor:
tensor([[[[1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.]]]])

Pre-computed freqs_cis:
tensor([[ 1.0000+0.0000j,  1.0000+0.0000j],
        [ 0.5403+0.8415j,  0.9999+0.0100j],
        [-0.4161+0.9093j,  0.9998+0.0200j],
        [-0.9900+0.1411j,  0.9996+0.0300j],
        [-0.6536-0.7568j,  0.9992+0.0400j],
        [ 0.2837-0.9589j,  0.9988+0.0500j],
        [ 0.9602-0.2794j,  0.9982+0.0600j],
        [ 0.7539+0.6570j,  0.9976+0.0699j]])

Rotated tensor:
tensor([[[[ 1.0000,  1.0000,  1.0000,  1.0000],
          [-0.3012,  1.3818,  0.9900,  1.0099],
          [-1.3254,  0.4932,  0.9798,  1.0198],
          [-1.1311, -0.8489,  0.9696,  1.0295],
          [ 0.1032, -1.4104,  0.9592,  1.0392],
          [ 1.2426, -0.6753,  0.9488,  1.0487],
          [ 1.2396,  0.6808,  0.9382,  1.0582],
          [ 0.0969,  1.4109,

## Multi-Head Latent Attention (MLA)

```
h_t  (B, T, 8)
 │
 ├─ W_DKV (8×4) ──→ c_kv (B,T,4)  ← cached at inference
 │    ├─ W_UK (4×8) ──→ K_nope (B,H,T,4)  ┐ per-head content
 │    └─ W_UV (4×8) ──→ V      (B,H,T,4)  ┘
 │
 ├─ W_DQ  (8×4) ──→ c_q  (B,T,4)
 │    ├─ W_UQ (4×8) ──→ Q_nope (B,H,T,4)          per-head content
 │    └─ W_QR (4×4) ──→ Q_rope (B,H,T,2) ──RoPE── per-head positional query
 │
 └─ W_KR  (8×2) ──→ K_rope (B,1,T,2) ──RoPE── SHARED positional key (broadcast)
                                                  ↓
             Q = [Q_nope ; Q_rope]  (B,H,T,6)   per-head
             K = [K_nope ; K_rope]  (B,H,T,6)   K_rope broadcast from (B,1,T,2)
             attn = softmax(Q·Kᵀ / √6) · V
```

`K_rope` is **head-shared**: `W_KR` outputs `d_head_rope` (not `n_heads × d_head_rope`).
The same positional key vector is appended to every head's key, saving `(n_heads - 1) × d_head_rope` in the KV cache per token.

In [ ]:
class MultiHeadLatentAttention(nn.Module):
    """
    DeepSeek-V2 style MLA with decoupled RoPE.
    """
    def __init__(self, d_model: int = D_MODEL, n_heads: int = N_HEADS, d_head_nope: int = D_HEAD_NOPE, d_head_rope: int = D_HEAD_ROPE,
                       d_c_kv: int = D_C_KV, d_c_q: int = D_C_Q, dropout: float = DROPOUT):
        super().__init__()
        self.n_heads     = n_heads
        self.d_head_nope = d_head_nope
        self.d_head_rope = d_head_rope
        self.d_head      = d_head_nope + d_head_rope
        self.scale       = self.d_head ** -0.5

        # ── KV pathway ───────────────────────────────────────────────────────
        self.W_DKV = nn.Linear(d_model, d_c_kv, bias=False)
        self.W_UK  = nn.Linear(d_c_kv,  n_heads*d_head_nope, bias=False)
        self.W_UV  = nn.Linear(d_c_kv,  n_heads*d_head_nope, bias=False)

        # ── Q pathway ────────────────────────────────────────────────────────
        self.W_DQ  = nn.Linear(d_model, d_c_q, bias=False)
        self.W_UQ  = nn.Linear(d_c_q,   n_heads*d_head_nope, bias=False)

        # ── Decoupled RoPE pathway ────────────────────────────────────────────
        self.W_QR = nn.Linear(d_c_q, n_heads*d_head_rope, bias=False)
        self.W_KR  = nn.Linear(d_model, d_head_rope, bias=False)  # HEAD-SHARED

        # ── Output ───────────────────────────────────────────────────────────
        self.W_O   = nn.Linear(n_heads*d_head_nope, d_model, bias=False)
        self.drop  = nn.Dropout(dropout)

    def forward(self, h: torch.Tensor, freqs_cis: torch.Tensor) -> torch.Tensor:
        """
        h         : (B, T, d_model)
        freqs_cis : (T, d_head_rope//2)  complex
        returns   : (B, T, d_model)
        """
        B, T, _ = h.shape

        # ── Step 1 · Compress ────────────────────────────────────────────────
        c_kv = self.W_DKV(h)   # (B, T, d_c_kv=4)
        c_q  = self.W_DQ(h)    # (B, T, d_c_q=4)

        # ── Step 2 · NoPE up-projection (per-head) ───────────────────────────
        def to_heads(x, d_h):
            return x.view(B, T, self.n_heads, d_h).transpose(1, 2)

        K_nope = to_heads(self.W_UK(c_kv), self.d_head_nope)  # (B, H, T, 4)
        V      = to_heads(self.W_UV(c_kv), self.d_head_nope)  # (B, H, T, 4)
        Q_nope = to_heads(self.W_UQ(c_q),  self.d_head_nope)  # (B, H, T, 4)

        # ── Step 3 · Decoupled RoPE pathway ──────────────────────────────────
        # Q_rope: per-head
        Q_rope = to_heads(self.W_QR(c_q), self.d_head_rope)
        Q_rope = apply_rope(Q_rope, freqs_cis)

        # K_rope: HEAD-SHARED — one vector per token, same for all heads
        K_rope = self.W_KR(h)                                    # (B, T, d_head_rope=2)
        K_rope = K_rope.unsqueeze(1)                             # (B, 1, T, 2)
        K_rope = apply_rope(K_rope, freqs_cis)                   # (B, 1, T, 2)  rotated
        K_rope = K_rope.expand(-1, self.n_heads, -1, -1)        # (B, H, T, 2)  broadcast

        # ── Step 4 · Concatenate NoPE + RoPE ─────────────────────────────────
        Q = torch.cat([Q_nope, Q_rope], dim=-1)  # (B, H, T, 6)
        K = torch.cat([K_nope, K_rope], dim=-1)  # (B, H, T, 6)

        # ── Step 5 · Causal attention ─────────────────────────────────────────
        attn = (Q @ K.transpose(-2, -1)) * self.scale
        mask = torch.tril(torch.ones(T, T, device=h.device, dtype=torch.bool))
        attn = attn.masked_fill(~mask, float("-inf"))
        attn = F.softmax(attn, dim=-1)
        attn = self.drop(attn)

        out = attn @ V                                            # (B, H, T, 4)
        out = out.transpose(1, 2).contiguous()
        out = out.view(B, T, self.n_heads * self.d_head_nope)
        return self.W_O(out)                                      # (B, T, d_model)

    def kv_cache_size_bytes(self, seq_len: int) -> dict:
        fp16 = 2
        # MLA caches: c_kv (shared) + K_rope (head-shared, NOT per-head)
        mla_total = (seq_len * D_C_KV + seq_len * self.d_head_rope) * fp16
        mha_total = seq_len * self.n_heads * self.d_head * fp16 * 2  # K + V
        return {
            "MHA  K+V per token": f"{self.n_heads}×{self.d_head} + {self.n_heads}×{self.d_head} = {2*self.n_heads*self.d_head} values",
            "MLA  c_kv + K_rope": f"{D_C_KV} + {self.d_head_rope} = {D_C_KV + self.d_head_rope} values  (K_rope shared, not ×{self.n_heads})",
            "bytes MHA": mha_total,
            "bytes MLA": mla_total,
            "compression_ratio": round(mha_total / mla_total, 2),
        }


mla = MultiHeadLatentAttention().to(DEVICE)
print(mla)
print()
for k, v in mla.kv_cache_size_bytes(SEQ_LEN).items():
    print(f"  {k}: {v}")

MultiHeadLatentAttention(
  (W_DKV): Linear(in_features=8, out_features=4, bias=False)
  (W_UK): Linear(in_features=4, out_features=8, bias=False)
  (W_UV): Linear(in_features=4, out_features=8, bias=False)
  (W_DQ): Linear(in_features=8, out_features=4, bias=False)
  (W_UQ): Linear(in_features=4, out_features=8, bias=False)
  (W_QR): Linear(in_features=4, out_features=4, bias=False)
  (W_KR): Linear(in_features=8, out_features=2, bias=False)
  (W_O): Linear(in_features=8, out_features=8, bias=False)
  (drop): Dropout(p=0.1, inplace=False)
)

  MHA  K+V per token: 2×6 + 2×6 = 24 values
  MLA  c_kv + K_rope: 4 + 2 = 6 values  (K_rope shared, not ×2)
  bytes MHA: 384
  bytes MLA: 96
  compression_ratio: 4.0


# MLA demo

In [44]:
mla.eval()
B = 1

h = torch.randn(B, SEQ_LEN, D_MODEL, device=DEVICE)
print(f"h (input) : {tuple(h.shape)}  (B, T, d_model)")

with torch.no_grad():
    c_kv = mla.W_DKV(h)
    c_q  = mla.W_DQ(h)
    print(f"\n── Step 1 · Compress ──────────────────────────────────────────")
    print(f"c_kv = h @ W_DKV           : {tuple(c_kv.shape)}")
    print(f"c_q  = h @ W_DQ            : {tuple(c_q.shape)}")

    def to_heads(x, d_h):
        return x.view(B, SEQ_LEN, N_HEADS, d_h).transpose(1, 2)

    K_nope = to_heads(mla.W_UK(c_kv), D_HEAD_NOPE)
    V      = to_heads(mla.W_UV(c_kv), D_HEAD_NOPE)
    Q_nope = to_heads(mla.W_UQ(c_q),  D_HEAD_NOPE)
    print(f"\n── Step 2 · NoPE up-projection (per-head) ─────────────────────")
    print(f"K_nope                     : {tuple(K_nope.shape)}  (B, H, T, d_head_nope)")
    print(f"V                          : {tuple(V.shape)}")
    print(f"Q_nope                     : {tuple(Q_nope.shape)}")

    Q_rope = apply_rope(to_heads(mla.W_QR(c_q), D_HEAD_ROPE), freqs_cis)

    K_rope_shared = mla.W_KR(h).unsqueeze(1)                  # (B, 1, T, 2)
    K_rope_shared = apply_rope(K_rope_shared, freqs_cis)       # rotate
    K_rope        = K_rope_shared.expand(-1, N_HEADS, -1, -1) # (B, H, T, 2)

    print(f"\n── Step 3 · Decoupled RoPE ─────────────────────────────────────")
    print(f"Q_rope  [per-head]         : {tuple(Q_rope.shape)}   (B, H, T, d_head_rope)")
    print(f"K_rope  [after W_KR]       : {tuple(mla.W_KR(h).shape)}  → unsqueeze → {tuple(K_rope_shared.shape)}")
    print(f"K_rope  [after broadcast]  : {tuple(K_rope.shape)}   same 2-dim vector in every head")

    Q = torch.cat([Q_nope, Q_rope], dim=-1)
    K = torch.cat([K_nope, K_rope], dim=-1)
    print(f"\n── Step 4 · Concat ─────────────────────────────────────────────")
    print(f"Q = [Q_nope ; Q_rope]      : {tuple(Q.shape)}  (B, H, T, {D_HEAD_NOPE}+{D_HEAD_ROPE})")
    print(f"K = [K_nope ; K_rope]      : {tuple(K.shape)}")

    attn = F.softmax(
        (Q @ K.transpose(-2, -1)) * D_HEAD**-0.5
        .masked_fill(~torch.tril(torch.ones(SEQ_LEN, SEQ_LEN, device=DEVICE, dtype=torch.bool)), float("-inf")),
        dim=-1,
    ) if False else None  # just show shapes
    print(f"\n── Step 5 · Attention ──────────────────────────────────────────")
    print(f"attn scores shape          : (B={B}, H={N_HEADS}, T={SEQ_LEN}, T={SEQ_LEN})")
    out = mla(h, freqs_cis)
    print(f"final output               : {tuple(out.shape)}  ")

    print(f"\n── KV-cache per token ──────────────────────────────────────────")
    print(f"  MHA: K (H×d_head) + V (H×d_head) = {N_HEADS}×{D_HEAD} + {N_HEADS}×{D_HEAD} = {2*N_HEADS*D_HEAD} values")
    print(f"  MLA: c_kv ({D_C_KV}) + K_rope ({D_HEAD_ROPE}, HEAD-SHARED) = {D_C_KV + D_HEAD_ROPE} values")
    print(f"       K_rope is NOT multiplied by n_heads={N_HEADS} — it is broadcast")

h (input) : (1, 8, 8)  (B, T, d_model)

── Step 1 · Compress ──────────────────────────────────────────
c_kv = h @ W_DKV           : (1, 8, 4)
c_q  = h @ W_DQ            : (1, 8, 4)

── Step 2 · NoPE up-projection (per-head) ─────────────────────
K_nope                     : (1, 2, 8, 4)  (B, H, T, d_head_nope)
V                          : (1, 2, 8, 4)
Q_nope                     : (1, 2, 8, 4)

── Step 3 · Decoupled RoPE ─────────────────────────────────────
Q_rope  [per-head]         : (1, 2, 8, 2)   (B, H, T, d_head_rope)
K_rope  [after W_KR]       : (1, 8, 2)  → unsqueeze → (1, 1, 8, 2)
K_rope  [after broadcast]  : (1, 2, 8, 2)   same 2-dim vector in every head

── Step 4 · Concat ─────────────────────────────────────────────
Q = [Q_nope ; Q_rope]      : (1, 2, 8, 6)  (B, H, T, 4+2)
K = [K_nope ; K_rope]      : (1, 2, 8, 6)

── Step 5 · Attention ──────────────────────────────────────────
attn scores shape          : (B=1, H=2, T=8, T=8)
final output               : (1, 8, 8)  

── 

## The Absorption Trick (inference optimisation)

During inference the NoPE content score is:

$$Q_{nope} \cdot K_{nope}^T = (c_Q \cdot W_{UQ}) \cdot (W_{UK} \cdot c_{KV})^T = c_Q \cdot \underbrace{(W_{UQ} \cdot W_{UK}^T)}_{W_{abs}} \cdot c_{KV}^T$$

Pre-compute $W_{abs}$ once → multiply the **low-rank** $c_Q$ and $c_{KV}$ directly. No need to ever materialise the full-dim $K_{nope}$.

In [43]:
# Weight matrices (rows = output, cols = input for nn.Linear, so .weight is (out, in))
# We need them as (in, out) for the math below
W_UQ = mla.W_UQ.weight.T   # (d_c_q,   n_heads*d_head_nope) = (4, 8)
W_UK = mla.W_UK.weight.T   # (d_c_kv,  n_heads*d_head_nope) = (4, 8)

print(f"W_UQ shape : {tuple(W_UQ.shape)}  (d_c_q  × n_heads*d_head_nope)")
print(f"W_UK shape : {tuple(W_UK.shape)}  (d_c_kv × n_heads*d_head_nope)")

# Pre-compute absorbed weight: (d_c_q, d_c_kv) — done ONCE before generation starts
W_abs = W_UQ @ W_UK.T        # (4, 4)
print(f"\nW_abs = W_UQ @ W_UK.T : {tuple(W_abs.shape)}  — precomputed once, lives on GPU")

# ── Naive route (what happens inside forward()) ───────────────────────────────
with torch.no_grad():
    c_q_demo  = mla.W_DQ(h)                             # (B, T, 4)
    c_kv_demo = mla.W_DKV(h)                            # (B, T, 4)

    Q_nope_naive = mla.W_UQ(c_q_demo)                   # (B, T, 8)
    K_nope_naive = mla.W_UK(c_kv_demo)                  # (B, T, 8)
    scores_naive = Q_nope_naive @ K_nope_naive.transpose(-2, -1)  # (B, T, T)

    # ── Absorbed route (skip materialising K_nope entirely) ──────────────────
    # c_q_demo @ W_abs → (B, T, d_c_kv),  then @ c_kv_demo.T → (B, T, T)
    scores_abs = (c_q_demo @ W_abs) @ c_kv_demo.transpose(-2, -1)  # (B, T, T)

    max_diff = (scores_naive - scores_abs).abs().max().item()
    print(f"\nMax difference between naive and absorbed scores: {max_diff:.2e}  (should be ~0)")

    print(f"\nMemory not allocated for K_nope during absorbed inference:")
    k_nope_bytes = B * SEQ_LEN * N_HEADS * D_HEAD_NOPE * 4   # float32
    print(f"  Saved {k_nope_bytes} bytes per forward pass (scales with batch×seq×heads)")

W_UQ shape : (4, 8)  (d_c_q  × n_heads*d_head_nope)
W_UK shape : (4, 8)  (d_c_kv × n_heads*d_head_nope)

W_abs = W_UQ @ W_UK.T : (4, 4)  — precomputed once, lives on GPU

Max difference between naive and absorbed scores: 1.79e-07  (should be ~0)

Memory not allocated for K_nope during absorbed inference:
  Saved 256 bytes per forward pass (scales with batch×seq×heads)
